<a href="https://colab.research.google.com/github/GabGP/Dermatology_AI_Project/blob/main/Dermatology_AI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0) **Libraries**

In [ ]:
# For neural networks
import tensorflow as tf

# For support vector machine
from sklearn.svm import SVC

# For random forest
from sklearn.ensemble import RandomForestClassifier

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# For dataset manipulation
import pandas as pd
import numpy as np
from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split

# For evaluation
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score, accuracy_score

# Miscellaneous Libraries
import os

# Global constant for training acceleration
AUTOTUNE = tf.data.AUTOTUNE

# 1) **Dataset Preparations**

In [ ]:
'''
The dataset to use:
https://archive.ics.uci.edu/dataset/33/dermatology
'''

!rm dermatology.data
!wget archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data

## Glimpse

In [ ]:
# Loading Dataset and having a glimpse of the dataset
column_names = [
    'erythema',
    'scaling',
    'definite-borders',
    'itching',
    'koebner phenomenon',
    'polygonal papules',
    'follicular papules',
    'oral-mucosal involvement',
    'knee elbow involvement',
    'scalp involvement',
    'family history',
    'melanin incontinence',
    'eosinophils in the infiltrate',
    'pnl infiltrate',
    'fibrosis of the papillary dermis',
    'exocytosis',
    'acanthosis',
    'hyperkeratosis',
    'parakeratosis',
    'clubbing of the rete ridges',
    'elongation of the rete ridges',
    'thinning of the suprapapillary epidermis',
    'spongiform pustule',
    'munro microabcess',
    'focal hypergranulosis',
    'disappearance of the granular layer',
    'vacuolisation and damage of the basal layer',
    'spongiosis',
    'saw-tooth appearance of retes',
    'follicular horn plug',
    'perifollicular parakeratosis',
    'inflammatory monoluclear infiltrate',
    'band-like infiltrate',
    'age',
    'class' # Target
     ];

raw_dataset = pd.read_csv(
    "dermatology.data",
    names=column_names,
    na_values="?",
    comment='\t',
    sep=',',
    skipinitialspace=True
)

raw_dataset.describe()


In [ ]:
# Taking a look at the top of the dataset
raw_dataset.head()

In [ ]:
# Checking how many columns and rows there are
raw_dataset.shape

In [ ]:
# Checking the types of each column
raw_dataset.dtypes

# 2) **Pre-Processing**

## Missing Values

In [ ]:
def missing_values_cleanup(dataset):
  new_dataset = dataset.copy()
  new_dataset = new_dataset.dropna()
  return new_dataset

# Checking for null values
print(raw_dataset.isna().sum())

clean_dataset = missing_values_cleanup(raw_dataset)

In [ ]:
clean_dataset.shape

## Encoding

The only variable that needs encoding is the target, everything else is already a numerical value and not a category.

In [ ]:
# Ordinal Enconding the Target
def encode_class(dataset):
  new_dataset = dataset.copy()
  new_dataset['class'] = new_dataset['class'] - 1
  return new_dataset

encoded_dataset = encode_class(clean_dataset)
encoded_dataset

## Separating Variables

### Correlation Matrix


In [ ]:
# Drawing the correlation matrix
correlation_matrix = encoded_dataset.corr()
plt.figure(figsize=(40, 15))
sns.heatmap(correlation_matrix, linewidths=.5, cmap='viridis', annot=True)
plt.title('Correlation Matrix - Dermatology Dataset')
plt.show()

### Dependent and Independent Variables


In [ ]:

# Selecting the dependent and independent variables based on the correlation matrix
dependent_variables = ['class']
independent_variables = [
    'erythema',
    'scaling',
    'definite-borders',
    'itching',
    'koebner phenomenon',
    'follicular papules',
    'family history',
    'pnl infiltrate',
    'exocytosis',
    'hyperkeratosis',
    'parakeratosis',
    'clubbing of the rete ridges',
    'spongiform pustule',
    'munro microabcess',
    'disappearance of the granular layer',
    'inflammatory monoluclear infiltrate',
    'age'
    ]

## Normalization


In [ ]:
def normalize_min_max(column):
    max_value = np.max(column)
    min_value = np.min(column)
    return (column - min_value) / (max_value - min_value)

In [ ]:
def normalize_columns(dataset, columns):
    new_dataset = dataset.copy()
    for column in columns:
        new_dataset[column] = normalize_min_max(new_dataset[column])
    return new_dataset

In [ ]:
columns_to_normalize = independent_variables
columns_to_normalize

## Balancing Classes

In [ ]:
# Drawing an histogram for the classes
for i in clean_dataset["class"].unique():
    numRows = len(clean_dataset[clean_dataset['class'] == i])
    print("Class", i, ": ", numRows)
sns.histplot(data=clean_dataset, x="class")
plt.title("Before Balancing")
plt.show()

In [ ]:
# Note: Not in use
def undersample_classes(dataset, target):
  new_dataset = dataset.copy()

  # Getting the amount of rows per classes
  values = new_dataset[target].value_counts()

  # Selecting the target class
  target_class = 0
  for index, value in values.items():
    if value == min(values):
      target_class = index

  reference_class_count = values[target_class]

  # Extracting all data from the target class
  undersampled_dataset = new_dataset[new_dataset[target] == target_class]

  classes = list(new_dataset[target].unique())
  classes.remove(target_class)

  for data_class in classes:
    class_data = new_dataset[new_dataset[target] == data_class].sample(reference_class_count, random_state=2026)
    undersampled_dataset = pd.concat([undersampled_dataset, class_data])

  # Shuffling
  undersampled_dataset = undersampled_dataset.sample(frac=1, random_state=2026).reset_index(drop=True)

  return undersampled_dataset

In [ ]:
def balance_classes(dataset, target):
    new_dataset = dataset.copy()

    # Calculate the median of the sample per class
    values = new_dataset[target].value_counts()
    reference_class_count = int(values.median())
    print(f"Median: {reference_class_count} samples")

    undersampled_parts = []

    for data_class in new_dataset[target].unique():
        class_data = new_dataset[new_dataset[target] == data_class]
        n = min(len(class_data), reference_class_count)
        undersampled_parts.append(class_data.sample(n, random_state=2026))
        print(f"Class {data_class}: {len(class_data)} → {n}")

    # Concatenating and shuffling
    undersampled_dataset = pd.concat(undersampled_parts)
    undersampled_dataset = undersampled_dataset.sample(frac=1, random_state=2026).reset_index(drop=True)

    return undersampled_dataset

In [ ]:
# Note: Testing the SMOTE-TOMEK method
def balance_classes_ST(dataset, target):
  new_dataset = dataset.copy()
  target_column = new_dataset[target]
  features_column = new_dataset.drop(target, axis=1)

  # Balancing classes with the SMOTE-TOMEK method
  smote_tomek = SMOTETomek(random_state=2026)
  values_set, values_target = smote_tomek.fit_resample(
      features_column,
      target_column)
  balanced_classes = pd.concat([values_set, values_target], axis=1)

  # Shuffling
  balanced_classes = balanced_classes.sample(frac=1, random_state=2026).reset_index(drop=True)

  return balanced_classes

In [ ]:
balanced_dataset = balance_classes_ST(clean_dataset, target='class')

# Drawing an histogram for the classes
for i in balanced_dataset["class"].unique():
    numRows = len(balanced_dataset[balanced_dataset['class'] == i])
    print("Class", i, ": ", numRows)
sns.histplot(data=balanced_dataset, x="class")
plt.title("After Balancing")
plt.show()

In [ ]:
balanced_dataset.describe()

## Executing Pre-Processing

In [ ]:
def pre_process_dataset(dataset, class_column, perform_balancing=False):
    # Cleaning up the dataset
    missingless_dataset = missing_values_cleanup(dataset=dataset)

    # Encoding the dataset
    encoded_dataset = encode_class(dataset=missingless_dataset)

    # Normalizing the dataset
    normalized_dataset = normalize_columns(dataset=encoded_dataset, columns=columns_to_normalize)

    # Balancing the dataset
    if perform_balancing:
        pre_processed_dataset = balance_classes_ST(
            dataset=normalized_dataset,
            target=class_column
        )
    else:
        pre_processed_dataset = normalized_dataset

    # Separating the variables
    values_set = pre_processed_dataset[independent_variables]
    values_target = pre_processed_dataset[dependent_variables]
    return values_set, values_target

In [ ]:
# Pre-Processing
full_set, full_target = pre_process_dataset(
    dataset=raw_dataset,
    class_column='class',
    perform_balancing=True
)

In [ ]:
# Splitting Dataset
train_set, test_set, train_target, test_target = train_test_split(
    full_set, full_target,
    test_size=0.2,
    random_state=2026
)

In [ ]:

train_target, test_target

# 3) **AI Model 1**

## Neural Network Classifier

### Architecture

In [ ]:
# Setting up the model
nnc_model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(len(independent_variables),)),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=6, activation='softmax')
])

# Compiling the model
nnc_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
    )

### Callbacks

In [ ]:
# Setting up EarlyStopping
earlystopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0.001,
    patience=5
)

In [ ]:
# Setting up ModelCheckpoint
checkpoint_path = "training/cp-{epoch:04d}.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True,
    save_freq='epoch'
  )

In [ ]:
# Setting up Tensorboard
%load_ext tensorboard
%mkdir logs & rm -rf ./logs/

import datetime
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

### Training

In [ ]:
# Training the model
nnc_model.fit(
    train_set,
    train_target,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        earlystopping_callback,
        tensorboard_callback,
        checkpoint_callback
        ]
)

#### Tensorboard

In [ ]:
# Visualizing the training process
%tensorboard --logdir logs/fit

## Evaluation

In [ ]:
# Evaluating the model
nnc_model.evaluate(test_set, test_target)

### Confusion Matrix

In [ ]:
# Getting the predictions
predictions = nnc_model.predict(test_set)
predicted_classes = np.argmax(predictions, axis=1)

# Getting the values for the confusion matrix
cm = confusion_matrix(test_target, predicted_classes)
r_score = recall_score(test_target, predicted_classes, average='macro')
p_score = precision_score(test_target, predicted_classes, average='macro')
fs = f1_score(test_target, predicted_classes, average='macro')

# Drawing the confusion matrix
plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=[1,2,3,4,5,6],
            yticklabels=[1,2,3,4,5,6])
plt.xlabel(f'Predicted - Precision: {p_score * 100:.4f}%')
plt.ylabel(f'Actual - Recall: {r_score * 100:.4f}%')
plt.title(f'Confusion Matrix - F1 Score: {fs * 100:.4f}%')
plt.show()

# 4) **AI Model 2**

## Neural Network Classifier

### Architecture

In [ ]:
# Setting up the model
model_2 = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(shape=(len(independent_variables),)),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=8, activation='relu'),
    tf.keras.layers.Dense(units=6, activation='softmax')
])

model_2.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer='adam',
    metrics=['accuracy']
    )

### Callbacks

In [ ]:
# Setting up EarlyStopping
earlystopping_callback_2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0.001,
    patience=5
)

In [ ]:
# Setting up Tensorboard
%load_ext tensorboard
%mkdir logs & rm -rf ./logs/

import datetime
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback_2 = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

### Training

In [ ]:
# Training the model
model_2.fit(
    train_set,
    train_target,
    epochs=150,
    batch_size=128,
    validation_split=0.2,
    callbacks=[earlystopping_callback_2, tensorboard_callback_2]
)

#### Tensorboard

In [ ]:
# Visualizing the training process
%tensorboard --logdir logs/fit

## Evaluation

In [ ]:
# Evaluating the model
model_2.evaluate(test_set, test_target)

### Confusion Matrix

In [ ]:
# Getting the predictions
predictions = model_2.predict(test_set)
predicted_classes = np.argmax(predictions, axis=1)

# Getting the values for the confusion matrix
cm = confusion_matrix(test_target, predicted_classes)
r_score = recall_score(test_target, predicted_classes, average='macro')
p_score = precision_score(test_target, predicted_classes, average='macro')
fs = f1_score(test_target, predicted_classes, average='macro')

# Drawing the confusion matrix
plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=[1,2,3,4,5,6],
            yticklabels=[1,2,3,4,5,6])
plt.xlabel(f'Predicted - Precision: {p_score * 100:.4f}%')
plt.ylabel(f'Actual - Recall: {r_score * 100:.4f}%')
plt.title(f'Confusion Matrix - F1 Score: {fs * 100:.4f}%')
plt.show()

# 5) **AI Model 3**

## Support Vector Classifier

### Architecture

In [ ]:
# Setting up the model
svc_model = SVC(C=1.0, kernel="rbf", decision_function_shape="ovo")

### Training

In [ ]:
# Training the model
svc_model.fit(train_set, train_target.values.ravel())

## Evaluation

In [ ]:
# Getting the predictions and accuracy
predicted_classes = svc_model.predict(test_set)
accuracy = accuracy_score(test_target, predicted_classes)
print(f'Accuracy: {accuracy * 100:.4f}%')

### Confusion Matrix

In [ ]:
# Getting the values for the confusion matrix
cm = confusion_matrix(test_target, predicted_classes)
r_score = recall_score(test_target, predicted_classes, average='macro')
p_score = precision_score(test_target, predicted_classes, average='macro')
fs = f1_score(test_target, predicted_classes, average='macro')

# Drawing the confusion matrix
plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=[1,2,3,4,5,6],
            yticklabels=[1,2,3,4,5,6])
plt.xlabel(f'Predicted - Precision: {p_score * 100:.4f}%')
plt.ylabel(f'Actual - Recall: {r_score * 100:.4f}%')
plt.title(f'Confusion Matrix - F1 Score: {fs * 100:.4f}%')
plt.show()

# 6) **AI Model 4**

## Random Forest Classifier

### Architecture

In [ ]:
# Setting up the model
rfc_model = RandomForestClassifier(n_estimators=100)

### Training

In [ ]:
# Training the model
rfc_model.fit(train_set, train_target.values.ravel())

## Evaluation

In [ ]:
# Getting the predictions and accuracy
predicted_classes = rfc_model.predict(test_set)
accuracy = accuracy_score(test_target, predicted_classes)
print(f'Accuracy: {accuracy * 100:.4f}%')

### Confusion Matrix

In [ ]:
# Getting the values for the confusion matrix
cm = confusion_matrix(test_target, predicted_classes)
r_score = recall_score(test_target, predicted_classes, average='macro')
p_score = precision_score(test_target, predicted_classes, average='macro')
fs = f1_score(test_target, predicted_classes, average='macro')

# Drawing the confusion matrix
plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=[1,2,3,4,5,6],
            yticklabels=[1,2,3,4,5,6])
plt.xlabel(f'Predicted - Precision: {p_score * 100:.4f}%')
plt.ylabel(f'Actual - Recall: {r_score * 100:.4f}%')
plt.title(f'Confusion Matrix - F1 Score: {fs * 100:.4f}%')
plt.show()